<a href="https://colab.research.google.com/github/satyaudaybandaru/Gemma-3-4B-FunctionCall/blob/main/Phase_1_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### In Phase 1 - We fine-tune exclusively on tool-calling data, effectively overfitting the model to tool usage

#### Installation

In [ ]:
%%capture
!pip install --upgrade unsloth transformers huggingface_hub trl

In [ ]:
from unsloth import FastModel, add_new_tokens
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
model.config.vocab_size = model.config.text_config.vocab_size ## For new transformer version compatability

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
# Verify Trainable Parameters

model.print_trainable_parameters()

#### Load Prepared Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_from_disk
filtered_dataset = load_from_disk("/content/drive/MyDrive/gemma3-dataset")

In [ ]:
filtered_dataset['text'][0]

#### Convert Text to tokens and mask labels

In [ ]:
MODEL_START = tk(
    "<start_of_turn>model\n",
    add_special_tokens=False
)["input_ids"]

END_TURN = tk(
    "<end_of_turn>",
    add_special_tokens=False
)["input_ids"]

print(MODEL_START)
print(END_TURN)

In [ ]:
def create_labels(input_ids):
    labels = [-100] * len(input_ids)

    i = 0

    while i < len(input_ids):

        if input_ids[i:i+3] == MODEL_START:

            start = i + 3

            j = start

            while j < len(input_ids):

                if input_ids[j:j+len(END_TURN)] == END_TURN:
                    labels[start:j+len(END_TURN)] = input_ids[start:j+len(END_TURN)]
                    i = j
                    break

                j += 1

        i += 1

    return labels

In [ ]:
def preprocess(example):
    encoded = tk(
        example["text"],
        truncation=True,
        max_length=4096,
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": create_labels(encoded["input_ids"]),
    }

In [ ]:
processed_dataset = filtered_dataset.map(
    preprocess,
    remove_columns=filtered_dataset.column_names,
    num_proc=4,
)

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tk,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tk,
    train_dataset = processed_dataset,
    eval_dataset = None, # Can set up evaluation!
    data_collator=data_collator,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 80,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = '/content/drive/MyDrive/gemma3-4b-toolcall/',
        save_steps = 10,
        save_total_limit = 7
    ),
)

Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [ ]:
tk.decode([tk.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tk.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# For transformer new version compatability

from transformers.models.gemma3.configuration_gemma3 import Gemma3Config

Gemma3Config.vocab_size = property(
    lambda self: self.text_config.vocab_size
)

tk.pad = tk.tokenizer.pad

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

**Continue From this in Phase-2 Training of Normal + Tool Call Convo**